In [1]:
import pandas as pd
import numpy as np

In [2]:
#Reading CSV File
path = r"C:\Users\Dimitris\Desktop\Online_Retail_Project.csv"

retail = pd.read_csv(path, encoding='utf-8-sig', sep=';')

In [3]:
#Check column's data types
retail.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  str    
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(1), int64(1), str(6)
memory usage: 69.3 MB


In [4]:
#Calculate NaN values for each column
retail.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [5]:
#Convert UnitPrice from str to float
retail['UnitPrice'] = retail['UnitPrice'].str.replace(',', '.').astype('float64')

In [6]:
#Convert InvoiceDate from str to datetime
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'], dayfirst=True)

In [7]:
#Replace NaN values from CustomerID column and convert them to int
retail = retail.dropna(subset = ['CustomerID'])
retail['CustomerID'] = retail['CustomerID'].astype('int64')

In [8]:
#Check column's data types again, and confirm the changes
retail.info()

<class 'pandas.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    406829 non-null  str           
 1   StockCode    406829 non-null  str           
 2   Description  406829 non-null  str           
 3   Quantity     406829 non-null  int64         
 4   InvoiceDate  406829 non-null  datetime64[us]
 5   UnitPrice    406829 non-null  float64       
 6   CustomerID   406829 non-null  int64         
 7   Country      406829 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(2), str(4)
memory usage: 47.9 MB


In [9]:
#Check again for NaN values
retail.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [10]:
#Count how many Quantity values are negative
(retail['Quantity'] < 0).sum()

np.int64(8905)

In [11]:
#Count how many UnitPrice values are negative or zero
(retail['UnitPrice'] <= 0).sum()

np.int64(40)

In [12]:
#Clearing the DataFrame and keep only the positive values of Quantity and UnitPrice
retail = retail[(retail['Quantity'] > 0) & (retail['UnitPrice'] > 0)]

In [13]:
#Create a new column to calculate the total amount of each product
retail['TotalAmount'] = retail['Quantity'] * retail['UnitPrice']

In [14]:
#Extract year,month,day,hour for further analysis
retail = retail.assign(Year = retail['InvoiceDate'].dt.year,
             Month = retail['InvoiceDate'].dt.month_name(),
             Year_Month = retail['InvoiceDate'].dt.strftime('%Y-%m'),
             Day_of_Week = retail['InvoiceDate'].dt.day_name(),
             Hour = retail['InvoiceDate'].dt.hour)

In [15]:
retail.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,Year,Month,Year_Month,Day_of_Week,Hour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010,December,2010-12,Wednesday,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010,December,2010-12,Wednesday,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010,December,2010-12,Wednesday,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010,December,2010-12,Wednesday,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010,December,2010-12,Wednesday,8


In [16]:
#Finding the top 10 customers
retail.groupby(['CustomerID'])['TotalAmount'].sum().sort_values(ascending = False).iloc[:10]

CustomerID
14646    280206.02
18102    259657.30
17450    194550.79
16446    168472.50
14911    143825.06
12415    124914.53
14156    117379.63
17511     91062.38
16029     81024.84
12346     77183.60
Name: TotalAmount, dtype: float64

In [17]:
#Top 10 selling products
retail.groupby(['Description'])['TotalAmount'].sum().sort_values(ascending = False).iloc[:10]

Description
PAPER CRAFT , LITTLE BIRDIE           168469.60
REGENCY CAKESTAND 3 TIER              142592.95
WHITE HANGING HEART T-LIGHT HOLDER    100448.15
JUMBO BAG RED RETROSPOT                85220.78
MEDIUM CERAMIC TOP STORAGE JAR         81416.73
POSTAGE                                77803.96
PARTY BUNTING                          68844.33
ASSORTED COLOUR BIRD ORNAMENT          56580.34
Manual                                 53779.93
RABBIT NIGHT LIGHT                     51346.20
Name: TotalAmount, dtype: float64

In [18]:
#Top 5 countries by total revenue
retail.groupby(['Country'])['TotalAmount'].sum().sort_values(ascending = False).head()

Country
United Kingdom    7308391.554
Netherlands        285446.340
EIRE               265545.900
Germany            228867.140
France             209024.050
Name: TotalAmount, dtype: float64

In [19]:
#Which month had the highest sales
retail.groupby(['Year_Month'])['TotalAmount'].sum().sort_values(ascending = False).head()

Year_Month
2011-11    1161817.380
2011-10    1039318.790
2011-09     952838.382
2011-05     678594.560
2011-06     661213.690
Name: TotalAmount, dtype: float64

In [20]:
#At what times of Day and Days of week do the most purchases take place
retail.groupby(['Hour','Day_of_Week'])[['InvoiceNo']].nunique().sort_values('InvoiceNo',ascending = False).head()

InvoiceNo
Hour Day_of_Week           
12   Wednesday          611
     Thursday           586
     Tuesday            531
13   Wednesday          493
12   Monday             480

In [21]:
#Create a new clear CSV file
retail.to_csv('Online_Retail_Cleaned.csv', index = False)